# 🚗 TakarMobil: Pengembangan Model Machine Learning (Phase 4)
*TakarMobil: Sistem Estimasi Rentang Harga & Analisis Faktor Penentu Harga Mobil Bekas*

---

### 🎯 Tujuan Tahap Pemodelan (Phase 4):
1. **Membangun Baseline Statistik:** Menetapkan acuan performa dasar (*lower-bound benchmark*) menggunakan estimasi median harga per merek dan model (`Merek_Model`).
2. **Melatih 4 Algoritma Regresi (dengan fitur `Model_Clean`):**
   - **Model 1: Ridge Regression** (Linear Regularized - L2)
   - **Model 2: Decision Tree Regressor** (Non-linear Tree Model)
   - **Model 3: Random Forest Regressor** (Ensemble Bagging)
   - **Model 4: XGBoost Regressor** (Ensemble Gradient Boosting)
3. **Hyperparameter Tuning:** Menerapkan optimasi parameter dengan validasi silang **5-Fold `KFold`** pada data training.
4. **Penyimpanan Seluruh Model:** Mengekspor seluruh model terbaik ke dalam format `.joblib` di folder `models/`.

## 1. Import Library & Setup Lingkungan
Memuat pustaka algoritma regresi, cross-validation, metrik evaluasi, serta pipeline scikit-learn.


In [1]:
import os
import json
import joblib
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("Seluruh pustaka pemodelan berhasil dimuat!")

Seluruh pustaka pemodelan berhasil dimuat!


## 2. Pemuatan Dataset Hasil Preprocessing (Train & Test)
Memuat `train_data.csv` (13.439 baris) dan `test_data.csv` (3.360 baris) yang telah disiapkan pada Phase 3.

In [2]:
candidate_train = ["data/processed/train_data.csv", "../data/processed/train_data.csv"]
candidate_test = ["data/processed/test_data.csv", "../data/processed/test_data.csv"]

train_path = next((p for p in candidate_train if os.path.exists(p)), None)
test_path = next((p for p in candidate_test if os.path.exists(p)), None)
if train_path is None or test_path is None:
    raise FileNotFoundError("Dataset train atau test tidak ditemukan!")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Data Training: {train_df.shape[0]} baris x {train_df.shape[1]} kolom")
print(f"Data Testing : {test_df.shape[0]} baris x {test_df.shape[1]} kolom")

num_cols = ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
cat_cols = ['Merek_Clean', 'Model_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']
all_features = num_cols + cat_cols

X_train = train_df[all_features]
y_train = train_df['Log_Harga']
y_train_raw = train_df['Harga_Num']

X_test = test_df[all_features]
y_test = test_df['Log_Harga']
y_test_raw = test_df['Harga_Num']

print(f"Fitur Numerik   : {num_cols}")
print(f"Fitur Kategorik : {cat_cols}")

Data Training: 13439 baris x 28 kolom
Data Testing : 3360 baris x 28 kolom
Fitur Numerik   : ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
Fitur Kategorik : ['Merek_Clean', 'Model_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']


## 3. Baseline Statistik (Median per Merek & Model)
Sebagai acuan dasar, Baseline Statistik memprediksi harga menggunakan median `Merek_Model` (dengan fallback median `Merek_Clean`) pada data training.

In [3]:
med_mm_log = train_df.groupby('Merek_Model')['Log_Harga'].median().to_dict()
med_m_log = train_df.groupby('Merek_Clean')['Log_Harga'].median().to_dict()

pred_baseline_log = np.array([med_mm_log.get(mm, med_m_log[m]) for mm, m in zip(test_df['Merek_Model'], test_df['Merek_Clean'])])
pred_baseline_raw = np.expm1(pred_baseline_log)

baseline_mae = mean_absolute_error(y_test_raw, pred_baseline_raw)
baseline_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_baseline_raw))
baseline_r2 = r2_score(y_test_raw, pred_baseline_raw)

print(f"=== HASIL EVALUASI BASELINE STATISTIK ===")
print(f"Skala Rupiah -> MAE: Rp {baseline_mae:,.0f} | RMSE: Rp {baseline_rmse:,.0f} | R2: {baseline_r2:.4f}")

=== HASIL EVALUASI BASELINE STATISTIK ===
Skala Rupiah -> MAE: Rp 61,975,400 | RMSE: Rp 176,658,558 | R2: 0.8136


## 4. Konfigurasi ColumnTransformer & Skema 5-Fold CV
- Fitur numerik distandarisasi dengan `StandardScaler`.
- Fitur kategorikal (`Merek_Clean`, `Model_Clean`, `Transmisi_Clean`, `Bahan_Bakar_Clean`, `Wilayah`) di-encode dengan `OneHotEncoder(handle_unknown='ignore')`.
- Validasi silang menggunakan **`KFold(n_splits=5, shuffle=True, random_state=42)`**.

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)
cv_split = KFold(n_splits=5, shuffle=True, random_state=42)
print("Preprocessor dan 5-Fold CV siap digunakan!")

Preprocessor dan 5-Fold CV siap digunakan!


## 5. Model 1: Ridge Regression (Linear Regularized - L2)
Ridge Regression menambahkan penalti L2 terhadap kuadrat bobot koefisien linier untuk mencegah multikolinearitas dan overfitting.
Kita mencari nilai optimal dari parameter regularisasi $\alpha$ (`alpha`).


In [5]:
start_time = time.time()
pipe_ridge = Pipeline([('preprocessor', preprocessor), ('model', Ridge())])
param_grid_ridge = {'model__alpha': [0.1, 1.0, 5.0, 10.0]}
grid_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=cv_split, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_ridge.fit(X_train, y_train)
ridge_time = time.time() - start_time
best_ridge = grid_ridge.best_estimator_
pred_ridge_raw = np.expm1(best_ridge.predict(X_test))
ridge_mae = mean_absolute_error(y_test_raw, pred_ridge_raw)
ridge_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_ridge_raw))
ridge_r2 = r2_score(y_test_raw, pred_ridge_raw)
print(f"Selesai dalam {ridge_time:.2f}s | Best: {grid_ridge.best_params_}")
print(f"Test Set -> MAE: Rp {ridge_mae:,.0f} | RMSE: Rp {ridge_rmse:,.0f} | R2: {ridge_r2:.4f}")

Selesai dalam 9.20s | Best: {'model__alpha': 0.1}
Test Set -> MAE: Rp 35,610,720 | RMSE: Rp 117,226,328 | R2: 0.9179


## 6. Model 2: Decision Tree Regressor
Decision Tree membagi ruang fitur secara bertingkat untuk menangkap hubungan non-linier antara usia, jarak tempuh, dan merek terhadap harga mobil.
Parameter yang dituning:
- `max_depth`: Kedalaman maksimum pohon untuk mengontrol kompleksitas.
- `min_samples_leaf`: Jumlah minimum sampel pada daun akhir pohon untuk mencegah overfitting.


In [6]:
start_time = time.time()
pipe_dt = Pipeline([('preprocessor', preprocessor), ('model', DecisionTreeRegressor(random_state=42))])
param_grid_dt = {'model__max_depth': [10, 16, 22], 'model__min_samples_leaf': [2, 5]}
grid_dt = GridSearchCV(pipe_dt, param_grid_dt, cv=cv_split, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_dt.fit(X_train, y_train)
dt_time = time.time() - start_time
best_dt = grid_dt.best_estimator_
pred_dt_raw = np.expm1(best_dt.predict(X_test))
dt_mae = mean_absolute_error(y_test_raw, pred_dt_raw)
dt_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_dt_raw))
dt_r2 = r2_score(y_test_raw, pred_dt_raw)
print(f"Selesai dalam {dt_time:.2f}s | Best: {grid_dt.best_params_}")
print(f"Test Set -> MAE: Rp {dt_mae:,.0f} | RMSE: Rp {dt_rmse:,.0f} | R2: {dt_r2:.4f}")

Selesai dalam 3.39s | Best: {'model__max_depth': 22, 'model__min_samples_leaf': 2}
Test Set -> MAE: Rp 31,387,556 | RMSE: Rp 199,223,374 | R2: 0.7630


## 7. Model 3: Random Forest Regressor (Ensemble Bagging)
Random Forest menggabungkan banyak pohon keputusan (*ensemble bagging*) dengan subsampling acak fitur dan data, menghasilkan prediksi yang stabil dan variansi rendah.
Parameter yang dituning:
- `n_estimators`: Jumlah pohon di dalam forest.
- `max_depth`: Kedalaman maksimum tiap pohon.
- `min_samples_leaf`: Batas minimum sampel per daun.


In [7]:
start_time = time.time()
pipe_rf = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])
param_grid_rf = {'model__n_estimators': [100], 'model__max_depth': [18, 24], 'model__min_samples_leaf': [2, 4]}
grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=cv_split, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)
rf_time = time.time() - start_time
best_rf = grid_rf.best_estimator_
pred_rf_raw = np.expm1(best_rf.predict(X_test))
rf_mae = mean_absolute_error(y_test_raw, pred_rf_raw)
rf_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_rf_raw))
rf_r2 = r2_score(y_test_raw, pred_rf_raw)
print(f"Selesai dalam {rf_time:.2f}s | Best: {grid_rf.best_params_}")
print(f"Test Set -> MAE: Rp {rf_mae:,.0f} | RMSE: Rp {rf_rmse:,.0f} | R2: {rf_r2:.4f}")

Selesai dalam 69.44s | Best: {'model__max_depth': 24, 'model__min_samples_leaf': 2, 'model__n_estimators': 100}
Test Set -> MAE: Rp 29,325,237 | RMSE: Rp 180,887,380 | R2: 0.8046


## 8. Model 4: XGBoost Regressor (Gradient Boosting)
XGBoost (*Extreme Gradient Boosting*) membangun pohon keputusan secara sekuensial (*boosting*), di mana setiap pohon baru mengoreksi residual kesalahan dari pohon sebelumnya secara matematis efisien.
Parameter yang dituning:
- `n_estimators`: Jumlah tahapan boosting.
- `max_depth`: Kedalaman pohon.
- `learning_rate` ($\eta$): Laju penyusutan kontribusi tiap pohon.
- `subsample`: Rasio sampel acak tiap iterasi boosting.


In [8]:
start_time = time.time()
pipe_xgb = Pipeline([('preprocessor', preprocessor), ('model', XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror'))])
param_grid_xgb = {'model__n_estimators': [150, 220], 'model__max_depth': [6, 8], 'model__learning_rate': [0.06, 0.1]}
grid_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=cv_split, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
xgb_time = time.time() - start_time
best_xgb = grid_xgb.best_estimator_
pred_xgb_raw = np.expm1(best_xgb.predict(X_test))
xgb_mae = mean_absolute_error(y_test_raw, pred_xgb_raw)
xgb_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_xgb_raw))
xgb_r2 = r2_score(y_test_raw, pred_xgb_raw)
print(f"Selesai dalam {xgb_time:.2f}s | Best: {grid_xgb.best_params_}")
print(f"Test Set -> MAE: Rp {xgb_mae:,.0f} | RMSE: Rp {xgb_rmse:,.0f} | R2: {xgb_r2:.4f}")

Selesai dalam 17.97s | Best: {'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__n_estimators': 220}
Test Set -> MAE: Rp 30,890,802 | RMSE: Rp 161,596,362 | R2: 0.8440


## 9. Tabel Perbandingan Evaluasi Seluruh Model
Tabel ringkasan metrik evaluasi (MAE, MSE, RMSE, R²) pada data testing (skala nominal Rupiah asli).


In [9]:
results_data = {
    'Model': ['Baseline Statistik (Median)', 'Ridge Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'MAE (Rupiah)': [baseline_mae, ridge_mae, dt_mae, rf_mae, xgb_mae],
    'RMSE (Rupiah)': [baseline_rmse, ridge_rmse, dt_rmse, rf_rmse, xgb_rmse],
    'R² Score': [baseline_r2, ridge_r2, dt_r2, rf_r2, xgb_r2],
    'Training Time (s)': [0.01, ridge_time, dt_time, rf_time, xgb_time]
}

df_results = pd.DataFrame(results_data)
# Hitung MSE
df_results['MSE (Rupiah)'] = df_results['RMSE (Rupiah)'] ** 2

# Urutkan berdasarkan R² tertinggi
df_results = df_results.sort_values(by='R² Score', ascending=False).reset_index(drop=True)

# Tampilkan Format Rapi
display_table = df_results.copy()
display_table['MAE (Rupiah)'] = display_table['MAE (Rupiah)'].apply(lambda x: f"Rp {x:,.0f}")
display_table['RMSE (Rupiah)'] = display_table['RMSE (Rupiah)'].apply(lambda x: f"Rp {x:,.0f}")
display_table['MSE (Rupiah)'] = display_table['MSE (Rupiah)'].apply(lambda x: f"{x:.2e}")
display_table['R² Score'] = display_table['R² Score'].apply(lambda x: f"{x:.4f}")
display_table['Training Time (s)'] = display_table['Training Time (s)'].apply(lambda x: f"{x:.2f} s")

print("=== TABEL PERBANDINGAN MODEL PADA TESTING SET ===")
display_table[['Model', 'MAE (Rupiah)', 'RMSE (Rupiah)', 'MSE (Rupiah)', 'R² Score', 'Training Time (s)']]


=== TABEL PERBANDINGAN MODEL PADA TESTING SET ===


,Model,MAE (Rupiah),RMSE (Rupiah),MSE (Rupiah),R² Score,Training Time (s)
0,Ridge Regression,"Rp 35,610,720","Rp 117,226,328",1.37e+16,0.9179,9.20 s
1,XGBoost,"Rp 30,890,802","Rp 161,596,362",2.61e+16,0.8440,17.97 s
2,Baseline Statistik (Median),"Rp 61,975,400","Rp 176,658,558",3.12e+16,0.8136,0.01 s
3,Random Forest,"Rp 29,325,237","Rp 180,887,380",3.27e+16,0.8046,69.44 s
4,Decision Tree,"Rp 31,387,556","Rp 199,223,374",3.97e+16,0.7630,3.39 s


## 10. Penyimpanan Model Terlatih (.joblib)
Menyimpan keempat model yang telah dioptimasi beserta berkas ringkasan pelatihan ke direktori `models/`.


In [10]:
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)

# 1. Simpan Model Pipeline Lengkap
joblib.dump(best_ridge, os.path.join(models_dir, 'ridge_model.joblib'))
joblib.dump(best_dt, os.path.join(models_dir, 'decision_tree_model.joblib'))
joblib.dump(best_rf, os.path.join(models_dir, 'random_forest_model.joblib'))
joblib.dump(best_xgb, os.path.join(models_dir, 'xgboost_model.joblib'))

print("Seluruh berkas model .joblib berhasil disimpan di folder models/:")
print("- ridge_model.joblib")
print("- decision_tree_model.joblib")
print("- random_forest_model.joblib")
print("- xgboost_model.joblib")

# 2. Simpan Ringkasan Hasil Training ke JSON
summary_dict = {
    'models_evaluated': ['Baseline', 'Ridge', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'best_model': df_results.iloc[0]['Model'],
    'best_r2': float(df_results.iloc[0]['R² Score']),
    'best_params': {
        'Ridge': grid_ridge.best_params_,
        'Decision Tree': grid_dt.best_params_,
        'Random Forest': grid_rf.best_params_,
        'XGBoost': grid_xgb.best_params_
    },
    'results': df_results.to_dict(orient='records')
}

with open(os.path.join(models_dir, 'training_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary_dict, f, indent=2)

print(f"Ringkasan pelatihan disimpan ke: {os.path.join(models_dir, 'training_summary.json')}")


Seluruh berkas model .joblib berhasil disimpan di folder models/:
- ridge_model.joblib
- decision_tree_model.joblib
- random_forest_model.joblib
- xgboost_model.joblib


Ringkasan pelatihan disimpan ke: ../models\training_summary.json
